# AuraNode — EfficientNet-B0 X-Ray Classifier
## Google Colab Free T4 GPU Training Notebook
**Dataset:** NIH Chest X-Ray (via Kaggle)  
**Output:** ONNX model for CPU inference on Render free tier  
**Target:** AUC-ROC ≥ 0.75

### Instructions
Run each cell **one by one** using the ▶ button. Do NOT use Run All.

In [ ]:
# Cell 1 — Check GPU and install dependencies
import subprocess

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '⚠ No GPU detected — go to Runtime > Change Runtime Type > T4 GPU')

!pip install -q kaggle timm onnx onnxruntime scikit-learn matplotlib seaborn
print('✓ All dependencies installed')

In [ ]:
# Cell 2 — Kaggle auth and dataset download
# When prompted, upload your kaggle.json file
# Get it from: kaggle.com > Account Settings > API > Create New Token
from google.colab import files
import os, shutil

print('Upload your kaggle.json file when the dialog appears:')
uploaded = files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('✓ Kaggle auth configured')

os.makedirs('/content/nih_data', exist_ok=True)
print('Downloading NIH Chest X-Ray sample dataset (5-10 mins)...')
!kaggle datasets download -d nih-chest-xrays/sample --path /content/nih_data
!unzip -q /content/nih_data/*.zip -d /content/nih_data/
print('✓ Dataset downloaded and extracted')

In [ ]:
# Cell 3 — Explore and prepare data
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

data_dir = Path('/content/nih_data/sample/images')
labels_path = '/content/nih_data/sample/sample_labels.csv'

df = pd.read_csv(labels_path)
print(f'Total records: {len(df)}')
print('\nLabel distribution (top 10):')
print(df['Finding Labels'].value_counts().head(10))

# Binary labels: Normal=0, Abnormal=1
df['label'] = df['Finding Labels'].apply(
    lambda x: 0 if x.strip() == 'No Finding' else 1
)
df['filepath'] = df['Image Index'].apply(
    lambda x: str(data_dir / x)
)

# Keep only files that exist
df = df[df['filepath'].apply(lambda p: Path(p).exists())]
print(f'\nValid files: {len(df)}')
print(f'Normal: {(df.label==0).sum()}, Abnormal: {(df.label==1).sum()}')

# Stratified 80/10/10 split
train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df['label']
)
print(f'\nSplit — Train:{len(train_df)} Val:{len(val_df)} Test:{len(test_df)}')
print('✓ Data prepared')

In [ ]:
# Cell 4 — Dataset class and DataLoaders
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class XRayDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['filepath']).convert('RGB')
        label = torch.tensor(row['label'], dtype=torch.long)
        return self.transform(img), label

BATCH = 32
train_loader = DataLoader(
    XRayDataset(train_df, train_tf),
    batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    XRayDataset(val_df, eval_tf),
    batch_size=BATCH, shuffle=False, num_workers=2
)
test_loader = DataLoader(
    XRayDataset(test_df, eval_tf),
    batch_size=BATCH, shuffle=False, num_workers=2
)
print(f'✓ DataLoaders ready | Batch size: {BATCH}')

In [ ]:
# Cell 5 — Model setup (EfficientNet-B0)
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=2)

# Phase 1: Freeze backbone, only train classifier head
for param in model.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True

model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / Total: {total:,}')
print('✓ Model ready (Phase 1: head only)')

In [ ]:
# Cell 6 — Training and evaluation functions
from sklearn.metrics import roc_auc_score

def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    total_loss = correct = total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            out  = model(imgs)
            loss = criterion(out, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        correct    += out.argmax(1).eq(labels).sum().item()
        total      += labels.size(0)
    return total_loss / len(loader), correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = correct = total = 0
    all_probs, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            with torch.cuda.amp.autocast():
                out  = model(imgs)
                loss = criterion(out, labels)
            probs       = torch.softmax(out, dim=1)[:, 1]
            total_loss += loss.item()
            correct    += out.argmax(1).eq(labels).sum().item()
            total      += labels.size(0)
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    auc = roc_auc_score(all_labels, all_probs)
    return total_loss / len(loader), correct / total, auc

print('✓ Training functions defined')

In [ ]:
# Cell 7 — Phase 1 Training (head only, 5 epochs)
criterion = torch.nn.CrossEntropyLoss()
scaler    = torch.cuda.amp.GradScaler()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)

print('=' * 60)
print('PHASE 1: Training classifier head only (5 epochs)')
print('=' * 60)

for epoch in range(1, 6):
    tr_loss, tr_acc          = train_one_epoch(model, train_loader, optimizer, criterion, scaler, device)
    val_loss, val_acc, val_auc = evaluate(model, val_loader, criterion, device)
    scheduler.step()
    print(f'Ep {epoch}/5 | Train loss={tr_loss:.4f} acc={tr_acc:.3f} | '
          f'Val loss={val_loss:.4f} acc={val_acc:.3f} AUC={val_auc:.3f}')

print('\n✓ Phase 1 complete')

In [ ]:
# Cell 8 — Phase 2 Fine-tuning (all layers, 10 epochs)
# Unfreeze ALL layers
for param in model.parameters():
    param.requires_grad = True

optimizer2 = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer2, T_max=10)

best_auc   = 0.0
best_epoch = 0

print('=' * 60)
print('PHASE 2: Full fine-tuning (10 epochs)')
print('=' * 60)

for epoch in range(1, 11):
    tr_loss, tr_acc            = train_one_epoch(model, train_loader, optimizer2, criterion, scaler, device)
    val_loss, val_acc, val_auc = evaluate(model, val_loader, criterion, device)
    scheduler2.step()

    marker = ''
    if val_auc > best_auc:
        best_auc   = val_auc
        best_epoch = epoch
        torch.save(model.state_dict(), 'best_model.pth')
        marker = ' ← BEST SAVED'

    print(f'Ep {epoch}/10 | Train loss={tr_loss:.4f} acc={tr_acc:.3f} | '
          f'Val loss={val_loss:.4f} acc={val_acc:.3f} AUC={val_auc:.3f}{marker}')

print(f'\n✓ Phase 2 complete')
print(f'Best Val AUC: {best_auc:.4f} at epoch {best_epoch}')

if best_auc < 0.70:
    print('⚠ WARNING: AUC below 0.70 — tell your engineer before proceeding')
else:
    print('✓ Model quality acceptable for production export')

In [ ]:
# Cell 9 — Test set evaluation and plots
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, roc_auc_score
)
import matplotlib.pyplot as plt
import seaborn as sns

# Load best checkpoint
model.load_state_dict(torch.load('best_model.pth', map_location=device))
model.eval()

all_preds, all_probs_t, all_labels_t = [], [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        out  = model(imgs)
        probs = torch.softmax(out, dim=1)[:, 1]
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_probs_t.extend(probs.cpu().numpy())
        all_labels_t.extend(labels.numpy())

test_auc = roc_auc_score(all_labels_t, all_probs_t)

print('=' * 60)
print('TEST SET RESULTS')
print('=' * 60)
print(classification_report(
    all_labels_t, all_preds,
    target_names=['Normal', 'Abnormal'], digits=4
))
print(f'Test AUC-ROC: {test_auc:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('AuraNode EfficientNet-B0 Evaluation', fontsize=13, fontweight='bold')

cm = confusion_matrix(all_labels_t, all_preds)
sns.heatmap(cm, annot=True, fmt='d', ax=axes[0], cmap='Blues',
            xticklabels=['Normal','Abnormal'],
            yticklabels=['Normal','Abnormal'])
axes[0].set_title('Confusion Matrix')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

fpr, tpr, _ = roc_curve(all_labels_t, all_probs_t)
axes[1].plot(fpr, tpr, color='#2563eb', linewidth=2, label=f'AUC = {test_auc:.3f}')
axes[1].plot([0,1],[0,1], '--', color='#94a3b8', linewidth=1, label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('evaluation_results.png', dpi=120, bbox_inches='tight')
plt.show()
print('✓ Plots saved')

In [ ]:
# Cell 10 — Export to ONNX format
import onnx
import onnxruntime as ort

model.cpu()
model.eval()

dummy       = torch.randn(1, 3, 224, 224)
export_path = 'efficientnet_b0_v1.onnx'

torch.onnx.export(
    model,
    dummy,
    export_path,
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input':  {0: 'batch_size'},
        'output': {0: 'batch_size'},
    },
)
print(f'✓ ONNX export complete: {export_path}')

# Verify
print('Verifying ONNX model...')
onnx_model = onnx.load(export_path)
onnx.checker.check_model(onnx_model)
print('✓ ONNX model structure is valid')

sess       = ort.InferenceSession(export_path, providers=['CPUExecutionProvider'])
test_input = np.random.randn(1, 3, 224, 224).astype(np.float32)
output     = sess.run(None, {'input': test_input})
print(f'✓ ONNX inference works | Output shape: {output[0].shape}')

file_mb = os.path.getsize(export_path) / (1024 * 1024)
print(f'✓ Model file size: {file_mb:.1f} MB')
print('\n🎉 Model is ready for production deployment!')

In [ ]:
# Cell 11 — Download model and results
import os
from google.colab import files

print('Starting downloads...')
print('⚠ Allow popups in your browser if nothing downloads')

files.download('efficientnet_b0_v1.onnx')
files.download('evaluation_results.png')

print('\n' + '=' * 60)
print('✓ DOWNLOAD COMPLETE')
print('=' * 60)
print()
print('NEXT STEPS:')
print('1. Go to Supabase Dashboard → Storage → ml-models bucket')
print('2. Upload efficientnet_b0_v1.onnx')
print('3. Copy the public URL of the uploaded file')
print('4. Go to Render → Your Service → Environment Variables')
print('5. Set ONNX_MODEL_URL = that public URL')
print()
print('Your AuraNode AI backend is ready! 🚀')